# W06a 실습 B · 튜닝, SVD와 웹 앱

**딥러닝응용I(추천시스템) · 동덕여자대학교 데이터사이언스전공 · 유원상 교수 · 2026-2**

2026년 10월 6일 · W06a · 주교재 4.4–4.6.

CPU 런타임에서 첫 셀부터 실행합니다. 파일 → Drive에 사본 저장으로 자신의 실습본을 만드세요.
설치 후 재시작 안내가 나오면 런타임을 재시작하고 첫 셀부터 다시 실행합니다.
개인 PC 경로나 파일 업로드 없이 공식 MovieLens를 자동 준비합니다. 네트워크 실패는 오류로 알립니다.
빈칸은 문제용 변수로 분리했습니다. 미완성 문제를 건너뛰어도 독립 실험은 실행됩니다.
자동 검사 통과가 자신의 모든 설명·빈칸이 정답임을 뜻하지는 않습니다.

목표: 검증으로 설정을 선택하고 최종 평가한 뒤, SVD와 MF의 차이를 앱으로 설명합니다.


## 실행 환경 준비

실습 A 실행 여부와 관계없이 첫 셀부터 진행합니다.


In [ ]:
import os, sys, subprocess
os.environ['GRADIO_ANALYTICS_ENABLED']='False'
IN_COLAB='google.colab' in sys.modules
COURSE_VERSION='2026-fall-w06a'
if IN_COLAB:
    subprocess.run([sys.executable,'-m','pip','install','-q',
        f'luna-recommender-course[apps] @ git+https://github.com/lunalab-ai/recommender.git@{COURSE_VERSION}',
        'gradio==6.26.0'],check=True)
import numpy as np
import pandas as pd
from IPython.display import display,Markdown
from luna_recsys.mf_sgd import MFSGD,sgd_step
from luna_recsys.model_comparison import comparison_toy,definition_link
from luna_recsys.datasets import prepare_movielens
from luna_recsys.mf_evaluation import (ThreeWaySplit,ValidatedMF,split_three_way,
    predict_rows,rating_metrics,evaluate_mf,fit_validated,search_mf,svd_reconstruct)
from luna_recsys.mf_evaluation_lab import (history_figure,validation_view,
    comparison_view,svd_view,build_evaluation_app)
print('Python',sys.version.split()[0],'고정 수업 버전',COURSE_VERSION)


## 정의 바로가기와 입력·출력

아래 링크는 설치한 고정 버전의 정의 행으로 연결됩니다. 함수 이름을 누르고 docstring부터 읽으세요.

| 함수 | 입력 | 출력과 주의점 |
|---|---|---|
| split_three_way | user_id/movie_id/rating 관측표 | train/validation/test 사본; 중복 쌍 거부 |
| fit_validated | train, validation, K·α·β·epochs | ValidatedMF; 최저 검증 상태 복원, 전체 history 유지 |
| search_mf | train, validation, 후보 | 선택 모델과 비교표; test 인수 없음 |
| evaluate_mf | 학습된 MFSGD, 관측표 | n/RMSE/MAE/새 ID 대체 건수; clipping 없음 |
| svd_reconstruct | 완전한 2D 행렬, rank | 인수·재구성·오차; NaN을 자동 대입하지 않음 |
| build_evaluation_app | train, validation, report_fn | Gradio Blocks; launch는 따로 실행 |

fit_validated는 기존 MFSGD의 초기화와 sgd_step을 재사용합니다. history의 마지막 행과 복원된 model의 상태는 다를 수 있습니다.


In [ ]:
apis=[MFSGD,MFSGD.fit,MFSGD.predict,MFSGD.explain,MFSGD.rmse,sgd_step,
      ThreeWaySplit,ValidatedMF,split_three_way,predict_rows,rating_metrics,
      evaluate_mf,fit_validated,search_mf,svd_reconstruct,prepare_movielens,
      comparison_toy,definition_link,history_figure,validation_view,
      comparison_view,svd_view,build_evaluation_app]
display(Markdown('\n'.join(f'- [{f.__qualname__}]({definition_link(f,COURSE_VERSION)})' for f in apis)))


## 1. 분할을 먼저 고정하기

최종 test를 제외한 train/validation만 search_mf에 전달합니다. 기본 네 후보를 실행한 뒤 후보를 추가하는 확장은 새로운 평가 계획으로 다룹니다.


In [ ]:
prepared=prepare_movielens('data/local/w06a',mode='real')
assert prepared.mode in {'movielens','local'}
print(prepared.description)
all_ratings=prepared.data.ratings
assert len(all_ratings)==100000
ratings=all_ratings.sample(n=20000,random_state=20261006).reset_index(drop=True)
split=split_three_way(ratings,seed=20261006)
print('빠른 모드: 20,000개 관측 표본, 전체 100K 결과가 아닙니다.')
print('train / validation / test:',len(split.train),len(split.validation),len(split.test))
print('실제 분할 방법:',split.methods)


### B1 · 실행 전 예상

K가 2에서 8로 늘면 학습·검증 오차가 각각 어떻게 달라질지 예상하세요. β를 크게 하면 반드시 좋아지는지도 적으세요. 이후 표의 수치를 근거로 설명을 수정합니다.


In [ ]:
b1_prediction=""
b1_after=""


In [ ]:
chosen,table=search_mf(split.train,split.validation,factors=(2,8),regularizations=(.02,.2),epochs=20)
display(table)
display(history_figure(chosen.history,chosen.best_epoch))
print('선택:',chosen.model.n_factors,chosen.model.regularization,chosen.best_epoch)
print('아직 test 성능을 보며 후보를 고르지 않았습니다.')


In [ ]:
assert len(table)==4
assert np.isclose(chosen.best_validation_rmse,table.validation_rmse.min())
assert np.isclose(evaluate_mf(chosen.model,split.validation)['rmse'],chosen.best_validation_rmse)
assert set(chosen.model.train_.index)==set(split.train.index)


## 2. 설정 확정 후 마지막 테스트 평가

현재 선택 모델을 그대로 사용합니다. train+validation 재학습은 하지 않습니다. 모든 테스트 행을 포함하고 새 ID 대체 건수를 보고합니다. 여기서 결과를 본 뒤 같은 test로 후보를 계속 바꾸지 않습니다.


In [ ]:
final_mf=evaluate_mf(chosen.model,split.test)
final_mean=rating_metrics(split.test.rating,np.repeat(split.train.rating.mean(),len(split.test)))
display(pd.DataFrame([dict(model='Train mean',**final_mean),dict(model='Selected MF',**final_mf)]))
print('데이터 20K 표본; 60/20/20 관측 분할; clipping 없음; 초기화 표준편차 0.1; seed 20261006.')


In [ ]:
assert final_mf['n']==4000
assert np.isfinite([final_mf['rmse'],final_mf['mae']]).all()
assert final_mf['fallback_rows']>=max(final_mf['unknown_user'],final_mf['unknown_item'])


### B2 · 디버깅: 선택한 epoch를 버리는 코드

검증에서 고른 뒤 `model = MFSGD(epochs=20).fit(train)`으로 새 모델을 만들어 평가했습니다. 선택된 객체로 평가하는 식을 작성하세요. 힌트: chosen.model은 선택 상태로 복원되어 있습니다.


In [ ]:
b2_fixed_code=""
b2_reason=""


## 3. SVD로 완전 행렬 재구성

특이값 6,4를 가진 작은 행렬입니다. rank 1은 모든 칸이 3, rank 2는 원본을 재구성합니다. 재구성 오차와 테스트 RMSE를 섞지 마세요.

$$A_k=U_k\Sigma_kV_k^\top.$$


In [ ]:
matrix=np.array([[5.,1.],[1.,5.]])
for rank in [1,2]:
    decomposition=svd_reconstruct(matrix,rank)
    print('rank',rank,'특이값',decomposition['s'],'Frobenius 오차',decomposition['frobenius_error'])
    display(pd.DataFrame(decomposition['reconstruction']))
text,frame,figure=svd_view(2,True)
print(text);display(frame);display(figure)


In [ ]:
full=svd_reconstruct(matrix,2);low=svd_reconstruct(matrix,1)
assert np.allclose(full['reconstruction'],matrix)
assert np.allclose(low['reconstruction'],3*np.ones((2,2)))
assert np.isclose(low['frobenius_error']**2,low['discarded_energy'])
assert np.isclose(frame.iloc[0,1],0,atol=1e-12)


### B3 · 빈칸: 버린 패턴의 오차

새 행렬 diag(4,2)의 rank 1 재구성과 Frobenius 오차를 예상하고 함수로 확인하는 코드를 적으세요. 힌트: 작은 특이값 항을 버립니다.


In [ ]:
b3_reconstruction=None
b3_error=None


## 4. 앱에 사용할 빠른 데이터

이미 분리한 train에서만 2,000개를 뽑아 앱용으로 다시 나눕니다. 앱은 학습 1,200개·검증 400개를 사용하고 최종 test 4,000개에는 접근하지 않습니다. 앱의 수치를 앞의 전체 20K 실험 결과로 부르지 않습니다.


In [ ]:
app_split=split_three_way(split.train.sample(n=2000,random_state=20261006).reset_index(drop=True))
print('앱 학습/검증 관측:',len(app_split.train),len(app_split.validation))


### B4 · 웹 앱 출력 직접 확장

report_fn(history,best_epoch)은 화면에 추가할 문장을 반환합니다. 선택 epoch의 validation_rmse − train_rmse를 계산해 문장에 넣으세요. 힌트: history.loc[history.epoch == best_epoch].iloc[0]. 입력→함수→문자열 출력의 연결을 확인합니다.


In [ ]:
def my_report(history,best_epoch):
    return '선택 epoch의 학습·검증 차이를 계산하여 이 문장을 수정하세요.'


## 5. 연결 검사와 실행

validation_view가 문자열·표·그림을 반환합니다. 앱의 버튼 이벤트도 이 순서대로 출력에 연결합니다. 먼저 작은 callback을 실행해 내 문장이 나오는지 확인합니다.


In [ ]:
message,history,figure=validation_view(app_split.train,app_split.validation,epochs=3,report_fn=my_report)
print(message);display(history);display(figure)
app=build_evaluation_app(app_split.train,app_split.validation,report_fn=my_report)


In [ ]:
assert len(history)==4 and figure.axes
assert str(my_report(history,int(history.loc[history.validation_rmse.idxmin(),'epoch']))) in message
assert app.config['dependencies']
print('W06A_B_RUNTIME_OK')


### B5 · 디버깅: 출력 연결 순서

callback 반환은 (문자열, 표, 그림)인데 outputs가 (문자열, 그림, 표)라면 어떤 문제가 생기나요? 올바른 연결 순서를 쓰고 소스의 run.click을 확인하세요.


In [ ]:
b5_outputs=[]
b5_reason=""


## 웹 앱 열기

학습과 검증 탭에서 설정 한 개를 바꾸고 곡선과 내 문장을 확인합니다. 4개 후보 비교를 실행하고, SVD 탭에서는 rank 1/2와 0 대입 반례를 비교합니다. 공유 주소는 이 런타임이 실행되는 동안만 유효합니다. 셀이 실행 중으로 보이는 것은 서버를 유지하기 때문일 수 있습니다.


In [ ]:
if IN_COLAB:
    app.launch(share=True,debug=False)
else:
    print('로컬 자동 검증: 앱 생성과 callback 검사 완료. 실제 화면은 app.launch()로 엽니다.')


## 점검 퀴즈

1. search_mf에 test 인수가 없는 이유는 무엇인가요?
2. 후보의 검증 차이가 0.000011이면 보편적인 우위를 주장할 수 있나요?
3. 테스트의 새 ID 행을 버리면 어떤 평가가 되나요?
4. 완전 SVD 재구성 오차 0은 미관측 예측 성공을 뜻하나요?
5. 내 문장은 어떤 입력과 callback을 거쳐 앱에 표시되나요?


## 기억할 점과 참고자료

학습은 모수를 바꾸고, 검증은 설정을 고르며, 테스트는 선택이 끝난 모델을 채점합니다.
최적 설정과 지표에는 데이터·분할·후보·대체 규칙을 함께 적으세요.

- 임일(2025), 주교재 4.4–4.6, 79–94쪽.
- [MovieLens 100K](https://grouplens.org/datasets/movielens/100k/)
- [평가와 교차검증](https://scikit-learn.org/stable/modules/cross_validation.html)
- [NumPy SVD](https://numpy.org/doc/stable/reference/generated/numpy.linalg.svd.html)
- [Surprise SVD](https://surprise.readthedocs.io/en/latest/matrix_factorization.html)

전체 100K 실행이나 여러 seed 실험은 확장입니다. 새 프로토콜을 정했다면 이전 테스트 결과를 보며 계속 최적화하지 말고 별도의 평가 계획을 먼저 정하세요.
